In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, message=".*escape sequence.*")

from textwrap import dedent
import httpx
import json
import requests
import pandas as pd
import numpy as np
import re
import requests
import uuid
from openai import OpenAI
import time
import os
from dotenv import load_dotenv, find_dotenv

from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, BaseOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


load_dotenv(find_dotenv(usecwd=True))

pubmed_api_key= '6892c4129cef143ff92d11533848d2e0d908'#os.getenv("PUBMED_API_KEY", '')
use = 'olm'

if use == 'hf':
    os.environ["BASE_URL"] = "https://router.huggingface.co/v1"
    os.environ["MODEL_NAME"] ='Qwen/Qwen3-32B'
    os.environ["OPENAI_API_KEY"] = os.getenv("HUGGINGFACE_HUB_TOKEN2")
    
elif use == 'olm':
    os.environ["BASE_URL"] = "http://localhost:11434/v1"
    os.environ["MODEL_NAME"] ='qwen3:14b'
    os.environ["OPENAI_API_KEY"] = 'hey'
    
else:
    os.environ["BASE_URL"] = "https://gigachat.devices.sberbank.ru/api/v1"
    os.environ["MODEL_NAME"] ='GigaChat-Max'
    
    url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
    # Создадим идентификатор UUID (36 знаков)
    rq_uid = str(uuid.uuid4())
    payload={
      'scope': 'GIGACHAT_API_PERS'
    }
    headers = {
      'Content-Type': 'application/x-www-form-urlencoded',
      'Accept': 'application/json',
      'RqUID': rq_uid,
      'Authorization': f'Basic {os.getenv("GIGACHAT_AUTH")}'
    }

    response = requests.request("POST", url, headers=headers, data=payload, verify=False)
    os.environ["OPENAI_API_KEY"] = response.json()['access_token']
    

os.environ['TEMPERATURE'] = '0'


# for importing trialmind, api_key should be set beforehand
from trialmind.pubmed import pmid2papers, PubmedAPIWrapper, pmid2biocxml, parse_bioc_xml, pmid2fulltext
from trialmind.api import StudyCharacteristicsExtraction, ScreeningCriteriaGeneration,\
                            LiteratureScreening, ScreeningCriteriaCTGeneration,\
                            CTScreening
from trialmind.retrievers import split_text_into_chunks
import extract
import time

%load_ext autoreload
%autoreload 2

In [132]:
from pydantic import BaseModel, validator, Field, conlist  # This is the new version
from typing import Dict, Literal
from markdown_pdf import MarkdownPdf
from markdown_pdf import Section
import ast
import time

In [2]:
#embeddings = OllamaEmbeddings(model="qwen3-embedding")
embeddings = OllamaEmbeddings(model="all-minilm")
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
    #persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

text_splitter=RecursiveCharacterTextSplitter(chunk_size=300, 
                                             chunk_overlap=100)

In [3]:
# preloading the model to speed up calcs
for i in ['qwen3:0.6b','qwen3:8b','qwen3:14b']:
    os.environ["MODEL_NAME"] =i

    client = OpenAI(
            base_url=os.getenv("BASE_URL"),
            api_key=os.getenv("OPENAI_API_KEY"),
            http_client=httpx.Client(verify=False)
        )
    response = client.chat.completions.create(
            model=os.getenv("MODEL_NAME"),
            messages=[{'role':'user','content':''}],  # Ensure messages is a list
            temperature=0,
        extra_body={"reasoning_effort": "none"}
        )

os.environ["MODEL_NAME"] = 'qwen3:0.6b'
start = time.time()
response = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=[{'role':'user','content':'hey'}],  # Ensure messages is a list
        temperature=0,
    extra_body={"reasoning_effort": "none"}
    )
end = time.time()
print(end - start)

response#.choices[0]#.message.content

2.2828001976013184


ChatCompletion(id='chatcmpl-366', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today? Let me know what you need! 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1773839441, model='qwen3:0.6b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=19, prompt_tokens=17, total_tokens=36, completion_tokens_details=None, prompt_tokens_details=None))

In [3]:
end = time.time()
# translation (as a task) is simple => we use a smaller llm to get results quicker
os.environ["MODEL_NAME"] ='qwen3:8b'

# get main info from .pdf
print('\nGETTING INFO FROM FILE')
file_path = "docs/LuC_213_L00_DL_edited_oncobox_ru.pdf"
fin_condition,treatements_eng,df2,fin_condition_ru,treatements_ru = extract.info_from_doc(file_path, with_ru=True)

end = time.time() - end
print(end)


GETTING INFO FROM FILE
11.322214365005493


In [5]:
fin_condition_ru,treatements_ru[0]

('Немелкоклеточный рак легкого', ' Афатиниб ')

In [135]:
start = time.time()

# translation (as a task) is simple => we use a smaller llm to get results quicker
os.environ["MODEL_NAME"] ='qwen3:8b'

# get main info from .pdf
print('\nGETTING INFO FROM FILE')
file_path = "docs/LuC_213_L00_DL_edited_oncobox_ru.pdf"
fin_condition,treatements_eng,df2,fin_condition_ru,treatements_ru = extract.info_from_doc(file_path, with_ru=True)

end = time.time()
print(end-start)

# using a larger llm for better results
#os.environ["MODEL_NAME"] ='qwen3:14b'
os.environ["MODEL_NAME"] ='qwen3:8b'
for treatement in treatements_eng[:1]:
    
    # ________________Get clinical trials
    print('\nGETTING CLINICAL TRIALS')
    all_studies = extract.get_clinicaltrials(f'''"{fin_condition}"''', 
                                          #' OR '.join(treatements_eng), 
                                           treatement,  
                                          max_studies=3)
    print(all_studies.shape)
    all_text = all_studies.briefTitle.fillna("") + ": " + all_studies.briefSummary.fillna("")
    api = CTScreening()
    end2 = time.time()
    print('getting c.t.:', end2-end)
    print('\nSCREENING CLINICAL TRIALS')
    ec_pred = api.run(
        population = f"Patients with {fin_condition} undergoing treatment with {treatement}",
        intervention = f"{treatement}",
        comparator = "",
        outcome = "",
        llm = os.getenv("MODEL_NAME"),
        criteria = [f"Does the trial focus on patients with '{fin_condition}'?",
                 f"Does the trial examine the use or sensitivity of '{treatement}' among main treatments?"
               ],
        papers = all_text.values.tolist(), # make for the top-100 for demo
    )
    all_studies[['s1','s2']] = [i.evaluations for i in ec_pred]
    all_studies[['r1','r2']] = [i.rationale for i in ec_pred]
    all_studies.to_csv(f"res_files/ctrials_all_df_{treatement.replace(' ','_')}_{fin_condition.replace(' ','_')}.csv", 
              index=False)
    end3 = time.time() 
    print('time:', end3-end2)
    print('\nEXTR RESULTS FROM CLINICAL TRIALS')
    ctrials_fin = extract.ctrials_res(ec_pred, all_studies)
    chosen_st = all_studies[(all_studies.screen_eval>=0)&(all_studies.hasResults==True)]
    chosen_st['res_with_part'] = chosen_st['outcomeMeasures'].apply(lambda x: [obj for obj in x \
                                                              if(obj.get('unitOfMeasure','').lower() in ['percentage of participants','participants'])])
    chosen_st = chosen_st[chosen_st.res_with_part.str.len()>0]
    chosen_st['res']= [i.model_dump_json() for i in ctrials_fin]
    chosen_st.to_csv(f"res_files/ctrials_res_df_{treatement.replace(' ','_')}_{fin_condition.replace(' ','_')}.csv", 
              index=False)
    print(ctrials_fin)
    end4 = time.time()
    print('time:', end4-end3)
    '''
    # ________________Get papers
    print('\nGETTING PAPERS')
    search_api = PubmedAPIWrapper()
        # page_size is the max number of records to return!!!! not pages!
    tmp_inputs = {
            "page_size": 5,
            "keyword_map": {'conditions':[fin_condition], 
                            'treatments':[treatement]
                           },
            "keywords": {
                "OPERATOR": 'AND'
            }
    }
    response = search_api.build_search_query_and_get_pmid(tmp_inputs, 
                                                          api_key=pubmed_api_key)
    print(response[0],len(response[0]), response[1])
    df_papers = pmid2papers(pmid_list=response[0], 
                        api_key=pubmed_api_key)
    papers = df_papers[0]["Title"] + ": " + df_papers[0]["Abstract"].fillna("") # important to fillna
    papers = papers.tolist()
    end5 = time.time()
    print('time:', end5-end4)
        # screening
    print('\nSCREENING PAPERS')
    api = LiteratureScreening()
    ec_predP = api.run(
        population = f"Patients with {fin_condition} undergoing treatment with {treatement}",
        intervention = f"{treatement}",
        comparator = "",
        outcome = "",
        llm = os.getenv("MODEL_NAME"),
        criteria = [f"Does the study focus on patients/models/cells with '{fin_condition}'?",
                     f"Does the study examine the use/effect/sensitivity of '{treatement}' among main treatments?", 
                    f"Does the study describe the effect of '{treatement}' treatment?"],#title_criteria + content_criteria,
        papers = papers, # make for the top-100 for demo
    )
    
    evalsP = [i.evaluations for i in ec_predP]
    word2int = {"YES": 1, "UNCERTAIN": 0, "NO": -1}
    new_evalsP = []
    for one_e in evalsP:
        new_evalsP.append([word2int.get(item, 0) for item in one_e ])
    new_evalsP = np.array(new_evalsP)  
    df_p_e = df_papers[0]
    df_p_e[['s1','s2','s3']] = new_evalsP
    df_p_e[['r1','r2','r3']] = [i.rationale for i in ec_predP]
    df_p_e.to_csv(f"res_files/papers_all_df_{treatement.replace(' ','_')}_{fin_condition.replace(' ','_')}.csv", 
              index=False)
    
    df_p_e = pd.read_csv(f"res_files/papers_all_df_{treatement.replace(' ','_')}_{fin_condition.replace(' ','_')}.csv", 
              )
    chosen_df = df_p_e[(df_p_e['s1']>=1)&(df_p_e['s2']>=0)&(df_p_e['s3']>=0)]
    print(chosen_df.shape[0])
    end6 = time.time()
    print('time:', end6-end5)
    
    #end6=time.time()
    # if there are papers left AFTER screening
    if chosen_df.shape[0]:
        # ________________To RAG
            # full texts
        pmid_list = chosen_df.PMID.values.tolist()
        #['41213063',#'26451310',]
        print('\nGETTING FULL TEXT PAPERS')
        res = pmid2fulltext(pmid_list, api_key=pubmed_api_key)
        res = [parse_bioc_xml(r) for r in res]

        # transform the parsed xml into paper content
        papers0 = []
        for parsed in res:
            paper_content = []
            for parsed_ in parsed["passage"]:
                paper_content.append(parsed_['content'])
            paper_content = "\n".join(paper_content)
            papers0.append(paper_content)

        chosen_df['FullText'] = ''
        chosen_df['FullText'] = papers0

        pmid_list = chosen_df.PMID.values.tolist()
        papers_ch = chosen_df.FullText.values
        docs =  [Document(page_content=i, 
                          metadata={"source": j}
                         ) for i,j in zip(papers_ch,pmid_list)]
        end7 = time.time()
        print('time:', end7-end6)
        print('\nEMBEDDING...')
        all_splits = text_splitter.split_documents(docs)
        document_ids = vector_store.add_documents(documents=all_splits)
        print('\nFIN EMBEDDING')
        end8 = time.time()
        print('time:', end8-end7)
        
        ii =len(pmid_list) #4
        print('\nEXTR RESULTS FROM PAPERS')
        api = StudyCharacteristicsExtraction()
        res_extracted = api.run(
            papers_inp=[pmid_list[:ii],papers_ch[:ii]],
            #fields=[f'The effectiveness of treating {fin_condition} with {treatements_eng[0]}',
            #       ],
            fields=[f'{treatement} effectiveness, string, The outcome of treating {fin_condition} with {treatement}'
                   ],
            llm=os.getenv("MODEL_NAME"),
            chunk_size=0,
            chunk_overlap=0,
            thinking=False,
            vector_store = vector_store,
        )
        papers_fin_df = pd.DataFrame([[i.fieldresult[0].value, 
                               i.fieldresult[0].source_id,
                              i.fieldresult[0]._cited_blocks,] for i in res_extracted 
                             ],
                            columns = ['result','idxs','citations'])
        papers_fin_df['id'] = pmid_list
        papers_fin_df['class']= [i.model_dump_json() for i in res_extracted]
        papers_fin_df.to_csv(f"res_files/papers_res_df_{treatement.replace(' ','_')}_{fin_condition.replace(' ','_')}.csv", 
                             index=False)
        end9 = time.time()
        print('time:', end9-end8)
        print('\nCOMBINING RESULTS FROM CLINICAL TRIALS')
        rr = extract.combine_res(fin_condition, treatement, 
                             res_extracted, pmid_list)
        end10 = time.time()
        print('time:', end10 - end9)
        print(rr)
    '''    
endf = time.time()
print('FUll time:', endf - start)

rr = extract.combine_res(fin_condition, treatement, 
                             res_extracted, pmid_list)
rr


GETTING INFO FROM FILE
10.1024489402771

GETTING CLINICAL TRIALS
(9, 10)
getting c.t.: 0.6015973091125488

SCREENING CLINICAL TRIALS

parsed_results in asynch [PaperEvaluation(evaluations=['NO', 'UNCERTAIN'], rationale=['The trial focuses on patients with NRG1 gene fusion-positive solid tumors, not specifically on Non-small cell lung cancer.', 'The trial mentions Afatinib as a treatment but does not clearly examine its use or sensitivity among main treatments.'])]

parsed_results in asynch [PaperEvaluation(evaluations=['YES', 'YES'], rationale=['The trial focuses on patients with Non-small cell lung cancer as specified in the description.', 'The trial examines the use of Afatinib as the first-line treatment, which is a main treatment for the population studied.'])]

parsed_results in asynch [PaperEvaluation(evaluations=['YES', 'NO'], rationale=['The trial focuses on patients with non-small cell lung cancer (NSCLC), as it specifically mentions NSCLC patients in the context of the T790M

'Afatinib is approved for first-line treatment in advanced EGFR mutation-positive NSCLC [[41736995]]. Afatinib showed potential in improving treatment outcomes for NSCLC patients [[41672191]]. Afatinib showed efficacy in older adult EGFR-mutant lung cancer patients [[41673455]].'

In [139]:
os.environ["MODEL_NAME"] ='qwen3:14b'
ff = extract.ctrials_res(ec_pred, all_studies)
ff

[ClinicalResult(population=14, time_frame='At the time of disease progression (median duration of 11.4 months from start of treatment)', outcomes=[Outcome(category_name='T790M Mutation on Progression Biopsy', outcome=4)]),
 ClinicalResult(population=14, time_frame='At the time of disease progression (median duration of 11.4 months from start of treatment)', outcomes=[Outcome(category_name='T790M Mutation on Progression Biopsy', outcome=4)]),
 ClinicalResult(population=14, time_frame='At the time of disease progression (median duration of 11.4 months from start of treatment)', outcomes=[Outcome(category_name='T790M Mutation on Progression Biopsy', outcome=4)]),
 ClinicalResult(population=14, time_frame='At the time of disease progression (median duration of 11.4 months from start of treatment)', outcomes=[Outcome(category_name='T790M Mutation on Progression Biopsy', outcome=4)]),
 ClinicalResult(population=14, time_frame='At the time of disease progression (median duration of 11.4 month

In [136]:
os.environ["MODEL_NAME"] ='qwen3:8b'
extract.gen_report(treatements_eng, fin_condition)

26.03286862373352


In [146]:
to_work.nctId

2    NCT01074177
3    NCT02285361
4    NCT00711594
5    NCT04179890
6    NCT02440854
7    NCT01085136
8    NCT02364609
Name: nctId, dtype: str

In [193]:
import numpy as np
from pydantic import BaseModel, validator, Field, conlist  # This is the new version
from typing import Dict, Literal




class Measure(BaseModel):
    measure_description: str = Field(description='Short description of what is being measured',
                                    max_length=200)
    measure_result: float = Field(description='Percent of participants')
    
class GroupMeasures(BaseModel):
    group_description: str = Field(description='Short group description',
                                  max_length=200)
    measures: list[Measure]

class Outcome(BaseModel):
    description: str = Field(description='Short description of an outcome',
                            max_length=200)
    population: int = Field(description='Total number of participants.')
    time_frame: str = Field(description='Time frame', max_length=200 )
    measures: list[GroupMeasures]
    
class Outcomes(BaseModel):
    outcomes: list[Outcome]
    
    
PROMPT_RES_EXTRACTION  = f'''
You are a clinical specialist analyzing clinical trial study reports. 
Your task is to to extract specific information as structured data.

# Reply Format: 
Return the information in the following JSON-format.
```
{Outcomes.model_json_schema()}
```
You MUST return ONLY valid JSON, Do NOT include any explanations, comments, or extra text.
'''

In [196]:
os.environ["MODEL_NAME"] ='qwen3:0.6b'

messages = []
for i in to_work.res_with_part.values:
    n_outcomes = len(i)
    messages.append([{'role':'system', 'content':PROMPT_RES_EXTRACTION+' \no_think'},
            {'role':'user', 'content':f"{i}"}])
    
response = client.chat.completions.parse(
    model=os.getenv('MODEL_NAME'),
    messages=messages[1],
    temperature=0,
    response_format=Outcomes,
    extra_body={"reasoning_effort": "none"}
)
fin = response.choices[0].message.parsed

for i in fin.outcomes:
    print(i,'\n')

description='Percentage of participants with Adverse Drug Reactions (ADRs).' population=1220 time_frame='From baseline (Visit 1) until last visit (the last follow-up visit a patient actually attended during the study), up to 1051 days.' measures=[GroupMeasures(group_description='GIOTRIF® was prescribed according to the local label and at the discretion of the treating physician. The physicians indicated doses and timing based on the current authorized label in Korea.', measures=[Measure(measure_description='Percentage of participants', measure_result=89.92)])] 

description='Progression-Free Survival (PFS) Rate at 48 Weeks.' population=1000 time_frame='From week 0 until week 48. Up to 48 weeks.' measures=[GroupMeasures(group_description='GIOTRIF® was prescribed according to the local label and at the discretion of the treating physician.', measures=[Measure(measure_description='Percentage of participants', measure_result=73.74)])] 

description='Percentage of Participants With Best Res

In [143]:
chosen = all_studies[(all_studies.screen_eval>=0)&(all_studies.hasResults==True)]
chosen['res_with_part'] = chosen['outcomeMeasures'].apply(lambda x: [obj for obj in x \
                                                          if(obj.get('unitOfMeasure','').lower() in ['percentage of participants','participants'])])

to_work = chosen[chosen.res_with_part.str.len()>0]
to_work.res_with_part.values[0]

[{'type': 'PRIMARY',
  'title': 'Number of Participants That Have a T790M Mutation on Their Progression Biopsy.',
  'populationDescription': 'The 14 participants that experienced disease progression and were also eligible to be biopsied.',
  'reportingStatus': 'POSTED',
  'paramType': 'COUNT_OF_PARTICIPANTS',
  'unitOfMeasure': 'Participants',
  'timeFrame': 'At the time of disease progression (median duration of 11.4 months from start of treatment)',
  'groups': [{'id': 'OG000',
    'title': 'BIBW 2992',
    'description': 'BIBW 2992: Taken orally once a day'}],
  'denoms': [{'units': 'Participants',
    'counts': [{'groupId': 'OG000', 'value': '14'}]}],
  'classes': [{'categories': [{'measurements': [{'groupId': 'OG000',
        'value': '4'}]}]}]},
 {'type': 'SECONDARY',
  'title': 'Response Rate',
  'description': 'The number of participants with either a complete response (CR) or partial response (PR) as assessed by Response Evaluation Criteria in Solid Tumors (RECIST v1.1)\n\n* C